In [136]:
#@title 🛢️📊 __STEP 1:__ Conectemos nuestra base de datos y comencemos a trabajar 🚀 { display-mode: "form" }

import pandas as pd
import sqlite3
import time
from IPython.core.magic import register_cell_magic
from IPython.display import display, HTML, clear_output
from tqdm.notebook import tqdm

# 1. Configuración de enlaces
datasets = {
    'tt_da_95_data_hospital_diagnoses': "https://raw.githubusercontent.com/hectorhormazabal-tt/TT-SQL-DA-95/refs/heads/master/diagnoses_dim.csv",
    'tt_da_95_data_hospital_appointments': "https://raw.githubusercontent.com/hectorhormazabal-tt/TT-SQL-DA-95/refs/heads/master/emergency_appointments.csv",
    'tt_da_95_data_hospital_patients': "https://raw.githubusercontent.com/hectorhormazabal-tt/TT-SQL-DA-95/refs/heads/master/patients_dim.csv"
}

# 2. Conexión a la base de datos en memoria
connector = sqlite3.connect(':memory:')

# 3. Proceso de carga con Barra de Progreso Verde
print("⬇️🗂️ Initiating data download and synchronization.....🔄⚙️")
summary_data = []

# Usamos tqdm para la barra visual
for name, url in tqdm(datasets.items(), desc="Cargando Tablas", bar_format='{l_bar}{bar:20}| {n_fmt}/{total_fmt}'):
    # Descarga
    df = pd.read_csv(url)

    # Conversión a SQL
    df.to_sql(name, connector, index=False, if_exists='replace')

    # Guardar info para el resumen
    summary_data.append({
        "Table route": f"{name}",
        "Row": f"{len(df):,}",
        "Columns": len(df.columns)
    })
    time.sleep(0.5) # Breve delay para que la barra sea visible

# 4. Palabra mágica %%sql
@register_cell_magic
def sql(line, cell):
    try:
        resultado = pd.read_sql(cell, connector)
        clear_output(wait=True)
        display(HTML("<b style='color: #4CAF50;'>✅ Query completed successfully:</b>"))
        return display(resultado)
    except Exception as e:
        clear_output(wait=True)
        display(HTML(f"<b style='color: #F44336;'>❌ Query execution failed:</b><br><code style='color: grey;'>{str(e)}</code>"))

# 5. Interfaz final para el alumno
clear_output()
display(HTML("<h2 style='color: #0636F9'>✨ Database initialized - Hospital data ✨</h2>"))
display(HTML("<p>Data has been successfully loaded. Please use these names for your queries:</p>"))

# Mostramos el resumen en una tabla estética
display(pd.DataFrame(summary_data))

print("\n ✨🏥 ALL DONE! YOU ARE READY TO PRACTICE YOUR QUERY SKILLS! 💊🌡️🩹💉🩺✨")

,Table route,Row,Columns
0,tt_da_95_data_hospital_diagnoses,15,6
1,tt_da_95_data_hospital_appointments,"800,000",11
2,tt_da_95_data_hospital_patients,"25,000",10



 ✨🏥 ALL DONE! YOU ARE READY TO PRACTICE YOUR QUERY SKILLS! 💊🌡️🩹💉🩺✨


In [ ]:
%%sql
PRAGMA table_info('tt_da_95_data_hospital_diagnoses');

,cid,name,type,notnull,dflt_value,pk
0,0,icd10_id,TEXT,0,None,0
1,1,diagnosis_name,TEXT,0,None,0
2,2,medical_specialty,TEXT,0,None,0
3,3,risk_score,INTEGER,0,None,0
4,4,is_contagious,INTEGER,0,None,0
5,5,avg_recovery_weeks,INTEGER,0,None,0


In [ ]:
%%sql
SELECT
*
FROM tt_da_95_data_hospital_appointments LIMIT 10

,appointment_id,patient_id,icd10_id,admission_date,triage_level,systolic_bp,diastolic_bp,heart_rate,temp_celsius,total_cost,hospital_branch
0,1,PAT-25-12835,ICD-110,2025-07-21 23:54:00,Yellow,134,104,82,36.9,3604.36,Centro
1,2,PAT-25-23799,ICD-115,2025-09-24 09:19:00,Blue,161,90,53,37.8,1672.30,Polanco
2,3,PAT-25-17072,ICD-101,2025-03-25 12:15:00,Red,160,101,102,36.6,81709.34,Polanco
3,4,PAT-25-02907,ICD-101,2025-08-21 21:54:00,Red,103,79,59,37.6,92072.24,Centro
4,5,PAT-25-22416,ICD-113,2025-12-29 13:26:00,Yellow,146,81,110,37.2,4178.57,Santa Fe
5,6,PAT-25-06217,ICD-102,2025-10-22 10:38:00,Orange,123,73,69,38.6,13783.14,Santa Fe
6,7,PAT-25-10256,ICD-102,2025-11-19 07:31:00,Orange,91,65,86,35.1,17013.61,Polanco
7,8,PAT-25-01891,ICD-102,2025-09-11 01:31:00,Orange,169,90,115,37.1,17962.29,Centro
8,9,PAT-25-21948,ICD-101,2025-01-23 13:51:00,Red,115,76,82,38.2,97996.36,Monterrey
9,10,PAT-25-03694,ICD-113,2025-01-04 22:51:00,Yellow,144,105,113,36.1,3830.98,Santa Fe


In [140]:
%%sql

PRAGMA table_info("tt_da_95_data_hospital_patients")

,cid,name,type,notnull,dflt_value,pk
0,0,patient_id,TEXT,0,None,0
1,1,full_name,TEXT,0,None,0
2,2,gender,TEXT,0,None,0
3,3,birth_date,TEXT,0,None,0
4,4,occupation,TEXT,0,None,0
5,5,state_mx,TEXT,0,None,0
6,6,blood_type,TEXT,0,None,0
7,7,height_m,REAL,0,None,0
8,8,weight_kg,REAL,0,None,0
9,9,insurance_provider,TEXT,0,None,0


In [151]:
%%sql
SELECT
    blood_type,
    gender,
    COUNT(DISTINCT patient_id) AS patients_count,
    -- Porcentaje respecto al total absoluto de la tabla
    ROUND(100.0 * COUNT(DISTINCT patient_id) / SUM(COUNT(DISTINCT patient_id)) OVER(), 2) AS pct_of_total,
    -- Porcentaje respecto al total de cada tipo de sangre
    ROUND(100.0 * COUNT(DISTINCT patient_id) / SUM(COUNT(DISTINCT patient_id)) OVER(PARTITION BY blood_type), 2) AS pct_by_blood_type
FROM tt_da_95_data_hospital_patients WHERE blood_type IN ('AB+', 'A+')
GROUP BY 1, 2;


,blood_type,gender,patients_count,pct_of_total,pct_by_blood_type
0,A+,,94,0.93,1.89
1,A+,F,2405,23.83,48.24
2,A+,M,2396,23.74,48.05
3,A+,NB,91,0.90,1.83
4,AB+,,113,1.12,2.21
5,AB+,F,2461,24.38,48.19
6,AB+,M,2441,24.19,47.80
7,AB+,NB,92,0.91,1.80
